# ML Operations Assignment: Data Versioning & Differential Privacy

## Part A: Data Versioning with DVC and Git-LFS

This notebook demonstrates:
1. Data versioning using **DVC** and **Git-LFS**
2. Building ML models by switching data versions through the versioning tool
3. Implementing Differential Privacy using **Opacus**

**Key Principle**: The data is stored and versioned in the tool. We switch versions by checking out from the tool, NOT by loading different CSV files into memory.

---

## Setup and Imports

In [7]:
# # Install required packages (run once)
# import sys
# !{sys.executable} -m pip install lakefs-client dvc pandas numpy scikit-learn matplotlib seaborn opacus torch

  Using cached dvc-3.67.1-py3-none-any.whl.metadata (17 kB)
  Using cached celery-5.6.3-py3-none-any.whl.metadata (23 kB)
  Using cached dvc_http-2.32.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached dvc_task-0.40.2-py3-none-any.whl.metadata (10.0 kB)
  Using cached gto-1.9.0-py3-none-any.whl.metadata (4.9 kB)
  Using cached scmrepo-3.6.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached asyncssh-2.22.0-py3-none-any.whl.metadata (9.5 kB)
  Using cached aiohttp_retry-2.9.1-py3-none-any.whl.metadata (8.8 kB)
  Using cached aiohttp-3.13.5-cp310-cp310-macosx_10_9_x86_64.whl.metadata (8.1 kB)
Using cached dvc-3.67.1-py3-none-any.whl (470 kB)
Using cached dvc_task-0.40.2-py3-none-any.whl (21 kB)
Using cached celery-5.6.3-py3-none-any.whl (451 kB)
Using cached gto-1.9.0-py3-none-any.whl (45 kB)
Using cached scmrepo-3.6.2-py3-none-any.whl (74 kB)
Using cached asyncssh-2.22.0-py3-none-any.whl (374 kB)
Using cached aiohttp_retry-2.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
import subprocess
import os
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Data file path - THIS IS THE SINGLE FILE THAT GETS VERSIONED
DATA_FILE = 'athletes.csv'

---
# Common Functions (Used for Both Tools)

In [9]:
def clean_data(data):
    """
    Clean the athletes dataset according to assignment requirements.
    """
    data = data.copy()
    
    # Remove rows with NaN in required columns
    data = data.dropna(subset=['region','age','weight','height','howlong','gender','eat',
                               'train','background','experience','schedule','howlong',
                               'deadlift','candj','snatch','backsq','experience',
                               'background','schedule','howlong'])
    
    # Drop unnecessary columns
    cols_to_drop = ['affiliate','team','name','athlete_id','fran','helen','grace',
                    'filthy50','fgonebad','run400','run5k','pullups','train']
    cols_to_drop = [c for c in cols_to_drop if c in data.columns]
    data = data.drop(columns=cols_to_drop)
    
    # Remove Outliers
    data = data[data['weight'] < 1500]
    data = data[data['gender'] != '--']
    data = data[data['age'] >= 18]
    data = data[(data['height'] < 96) & (data['height'] > 48)]
    
    data = data[(data['deadlift'] > 0) & 
                (((data['gender'] == 'Male') & (data['deadlift'] <= 1105)) | 
                 ((data['gender'] == 'Female') & (data['deadlift'] <= 636)))]
    data = data[(data['candj'] > 0) & (data['candj'] <= 395)]
    data = data[(data['snatch'] > 0) & (data['snatch'] <= 496)]
    data = data[(data['backsq'] > 0) & (data['backsq'] <= 1069)]
    
    # Clean Survey Data
    decline_dict = {'Decline to answer|': np.nan}
    data = data.replace(decline_dict)
    data = data.dropna(subset=['background','experience','schedule','howlong','eat'])
    
    return data


def prepare_for_ml(data, test_size=0.2, random_state=42):
    """
    Calculate total_lift, encode features, and split into train/test.
    """
    data = data.copy()
    
    # Calculate total_lift
    data['total_lift'] = data['deadlift'] + data['candj'] + data['snatch'] + data['backsq']
    
    # Encode categorical variables
    df_ml = data.copy()
    categorical_cols = df_ml.select_dtypes(include=['object']).columns.tolist()
    
    label_encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        df_ml[col] = le.fit_transform(df_ml[col].astype(str))
        label_encoders[col] = le
    
    # Features exclude individual lifts (since total_lift is derived from them)
    exclude_cols = ['total_lift', 'deadlift', 'candj', 'snatch', 'backsq']
    feature_cols = [c for c in df_ml.columns if c not in exclude_cols]
    
    X = df_ml[feature_cols]
    y = df_ml['total_lift']
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    
    return X_train, X_test, y_train, y_test, data


def run_eda(df, version_name):
    """Run EDA on the dataset."""
    print(f"\n{'='*60}")
    print(f"EDA for {version_name}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nMissing Values: {df.isnull().sum().sum()}")
    print(f"\nNumeric Statistics:")
    print(df.describe())
    
    if 'total_lift' in df.columns:
        print(f"\nTotal Lift - Mean: {df['total_lift'].mean():.2f}, Std: {df['total_lift'].std():.2f}")
    
    return df


def train_and_evaluate(X_train, X_test, y_train, y_test, model, model_name):
    """Train model and return metrics."""
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    metrics = {
        'model': model_name,
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
        'mae': mean_absolute_error(y_test, y_pred),
        'r2': r2_score(y_test, y_pred)
    }
    
    print(f"\n{model_name}:")
    print(f"  RMSE: {metrics['rmse']:.2f}")
    print(f"  MAE:  {metrics['mae']:.2f}")
    print(f"  R2:   {metrics['r2']:.4f}")
    
    return metrics, model, scaler

---
# ========================================================
# TOOL 1: DVC (Data Version Control)
# ========================================================

DVC tracks data files alongside Git. The data file is the SAME file - we switch versions using `git checkout` + `dvc checkout`.

## DVC Setup (Run in Terminal First)

```bash
# Navigate to project directory
cd "Machine Learning Operations"

# Initialize Git and DVC (run once)
git init
dvc init

# Add to DVC tracking (creates athletes_data.csv.dvc)
dvc add athletes.csv
git add athletes.csv.dvc .gitignore
git commit -m "Add athletes data V1 (raw)"
git tag v1
```

In [10]:
# DVC Helper Functions
def dvc_checkout(version_tag):
    """Switch to a specific data version using DVC."""
    try:
        # Checkout git tag (which has the .dvc file for that version)
        subprocess.run(['git', 'checkout', version_tag], check=True, capture_output=True)
        # Checkout the actual data file from DVC cache
        subprocess.run(['dvc', 'checkout'], check=True, capture_output=True)
        print(f"Switched to data version: {version_tag}")
        return True
    except subprocess.CalledProcessError as e:
        print(f"Error: {e.stderr.decode() if e.stderr else e}")
        return False

def dvc_load_data():
    """Load the currently checked-out data version."""
    return pd.read_csv(DATA_FILE)

## DVC: Step 1 - Work with V1 (Raw Data)

In [12]:
# Checkout V1 from DVC
dvc_checkout('v1')

# For initial setup, we start with the raw data
# Load current data (V1 - raw)
print("Loading V1 (raw) data from DVC...")
df_v1 = dvc_load_data()  # Initial load before DVC is set up
# After DVC setup: df_v1 = dvc_load_data()

print(f"V1 Shape: {df_v1.shape}")

Switched to data version: v1
Loading V1 (raw) data from DVC...
V1 Shape: (423006, 27)


In [13]:
# EDA on V1
run_eda(df_v1, "V1 (Raw Data)")


EDA for V1 (Raw Data)
Shape: (423006, 27)

Missing Values: 7643954

Numeric Statistics:
          athlete_id            age        height         weight  \
count  423003.000000  331110.000000  1.598690e+05  229890.000000   
mean   292748.166538      32.516750  1.206217e+02     170.896137   
std    184969.660327       7.730671  2.097995e+04      58.379799   
min        82.000000      13.000000  0.000000e+00       1.000000   
25%    135091.500000      27.000000  6.600000e+01     145.000000   
50%    275839.000000      31.000000  6.900000e+01     170.000000   
75%    473188.000000      37.000000  7.200000e+01     192.000000   
max    633083.000000     125.000000  8.388607e+06   20175.000000   

               fran         helen         grace      filthy50      fgonebad  \
count  5.542600e+04  3.027900e+04  4.074500e+04  1.935900e+04  2.973800e+04   
mean   9.886691e+02  1.207950e+03  5.766025e+02  2.127863e+03  1.472252e+03   
std    7.200430e+04  6.824091e+04  4.891145e+04  6.055021e+04

,athlete_id,name,region,team,affiliate,gender,age,height,weight,fran,...,snatch,deadlift,backsq,pullups,eat,train,background,experience,schedule,howlong
0,2554.0,Pj Ablang,South West,Double Edge,Double Edge CrossFit,Male,24.0,70.0,166.0,NaN,...,NaN,400.0,305.0,NaN,NaN,I workout mostly at a CrossFit Affiliate|I hav...,I played youth or high school level sports|I r...,I began CrossFit with a coach (e.g. at an affi...,I do multiple workouts in a day 2x a week|,4+ years|
1,3517.0,Derek Abdella,NaN,NaN,NaN,Male,42.0,70.0,190.0,NaN,...,NaN,NaN,NaN,NaN,NaN,I have a coach who determines my programming|I...,I played youth or high school level sports|,I began CrossFit with a coach (e.g. at an affi...,I do multiple workouts in a day 2x a week|,4+ years|
2,4691.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5164.0,Abo Brandon,Southern California,LAX CrossFit,LAX CrossFit,Male,40.0,67.0,NaN,211.0,...,200.0,375.0,325.0,25.0,I eat 1-3 full cheat meals per week|,I workout mostly at a CrossFit Affiliate|I hav...,I played youth or high school level sports|,I began CrossFit by trying it alone (without a...,I usually only do 1 workout a day|,4+ years|
4,5286.0,Bryce Abbey,NaN,NaN,NaN,Male,32.0,65.0,149.0,206.0,...,150.0,NaN,325.0,50.0,I eat quality foods but don't measure the amount|,I workout mostly at a CrossFit Affiliate|I inc...,I played college sports|,I began CrossFit by trying it alone (without a...,I usually only do 1 workout a day|I strictly s...,1-2 years|
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
423001,574489.0,Odo Renata,Latin America,Team Guarujá Inox,CrossFit Guaruja,Female,36.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
423002,585696.0,Lozzie Trevor,Australia,FBP CrossFit Games Team,FBP CrossFit,Female,27.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
423003,608828.0,Marisol Smith,North West,CrossFit Oak Harbor,CrossFit Oak Harbor,Female,44.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
423004,628881.0,Pedrini Morgane,Europe,NaN,CrossFit 67,Female,20.0,64.0,61.0,NaN,...,NaN,80.0,143.0,NaN,I eat quality foods but don't measure the amount|,I workout mostly at a CrossFit Affiliate|,NaN,I began CrossFit with a coach (e.g. at an affi...,I usually only do 1 workout a day|I strictly s...,6-12 months|


In [14]:
# Prepare V1 for ML (minimal cleaning to have valid lifts)
df_v1_clean = df_v1.dropna(subset=['deadlift', 'candj', 'snatch', 'backsq'])
df_v1_clean = df_v1_clean[(df_v1_clean['deadlift'] > 0) & 
                          (df_v1_clean['candj'] > 0) & 
                          (df_v1_clean['snatch'] > 0) & 
                          (df_v1_clean['backsq'] > 0)]

X_train_v1, X_test_v1, y_train_v1, y_test_v1, df_v1_full = prepare_for_ml(df_v1_clean)
print(f"V1 Train: {len(X_train_v1)}, Test: {len(X_test_v1)}")

V1 Train: 65492, Test: 16374


In [ ]:
# Train models on V1
print("="*60)
print("DVC - TRAINING ON V1 (RAW DATA)")
print("="*60)

dvc_v1_results = {}

dvc_v1_results['rf'], _, _ = train_and_evaluate(
    X_train_v1, X_test_v1, y_train_v1, y_test_v1,
    RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE), "Random Forest"
)

DVC - TRAINING ON V1 (RAW DATA)

Random Forest:
  RMSE: 104087.33
  MAE:  2013.66
  R2:   -1.4852


## DVC: Step 2 - Create and Commit V2 (Cleaned Data)

Now we clean the data, overwrite the SAME file, and commit as V2.

In [33]:
# Clean the data and save as V2 (overwriting the same file)
df_v2 = clean_data(df_v1)
df_v2.to_csv(DATA_FILE, index=False)
print(f"Saved cleaned data to {DATA_FILE}")
print(f"V2 Shape: {df_v2.shape}")
print(f"Rows removed from V1: {len(df_v1) - len(df_v2)}")

Saved cleaned data to athletes.csv
V2 Shape: (30015, 14)
Rows removed from V1: 392991


### Commit V2 to DVC (Run in Terminal)

```bash
# The file has changed, update DVC tracking
dvc add athletes.csv
git add athletes.csv.dvc
git commit -m "Update to athletes data V2 (cleaned)"
git tag v2
```

## DVC: Step 3 - Switch to V2 and Train (SAME CODE)

In [15]:
# Switch to V2 using DVC
dvc_checkout('v2')

# Load data - THE SAME LOAD COMMAND, but now it's V2
print("Loading V2 (cleaned) data from DVC...")
df = dvc_load_data()  # Same command as V1!
print(f"Loaded data shape: {df.shape}")

Switched to data version: v2
Loading V2 (cleaned) data from DVC...
Loaded data shape: (30015, 14)


In [16]:
# EDA on V2 - SAME CODE as V1
run_eda(df, "V2 (Cleaned Data)")


EDA for V2 (Cleaned Data)
Shape: (30015, 14)

Missing Values: 0

Numeric Statistics:
                age        height        weight         candj        snatch  \
count  30015.000000  30015.000000  30015.000000  30015.000000  30015.000000   
mean      32.125271     68.873397    177.146660    205.427520    156.178011   
std        7.436255      3.774809     32.397735     58.696774     48.841798   
min       18.000000     52.000000      5.000000      1.000000      1.000000   
25%       27.000000     66.000000    155.000000    160.000000    120.000000   
50%       31.000000     69.000000    178.000000    205.000000    155.000000   
75%       37.000000     72.000000    197.000000    245.000000    190.000000   
max       56.000000     83.000000    474.000000    390.000000    386.000000   

           deadlift        backsq  
count  30015.000000  30015.000000  
mean     362.102815    294.141596  
std       96.451618     85.090633  
min        1.000000      1.000000  
25%      287.000000   

,region,gender,age,height,weight,candj,snatch,deadlift,backsq,eat,background,experience,schedule,howlong
0,Southern California,Male,30.0,71.0,200.0,235.0,175.0,385.0,315.0,I eat whatever is convenient|,I played youth or high school level sports|I p...,I began CrossFit by trying it alone (without a...,I do multiple workouts in a day 1x a week|I ty...,1-2 years|
1,Africa,Male,28.0,70.0,176.0,187.0,134.0,335.0,254.0,I eat 1-3 full cheat meals per week|,I have no athletic background besides CrossFit|,I began CrossFit with a coach (e.g. at an affi...,I do multiple workouts in a day 1x a week|,2-4 years|
2,North East,Male,35.0,68.0,225.0,285.0,205.0,440.0,405.0,I eat quality foods but don't measure the amount|,I played youth or high school level sports|,I began CrossFit with a coach (e.g. at an affi...,I typically rest 4 or more days per month|,2-4 years|
3,North Central,Male,36.0,71.0,199.0,267.0,212.0,485.0,390.0,I eat quality foods but don't measure the amount|,I played youth or high school level sports|I p...,I began CrossFit with a coach (e.g. at an affi...,I do multiple workouts in a day 3+ times a wee...,1-2 years|
4,North East,Male,36.0,64.0,155.0,245.0,180.0,415.0,385.0,I eat strict Paleo|,I played youth or high school level sports|I p...,I began CrossFit by trying it alone (without a...,I do multiple workouts in a day 2x a week|I st...,4+ years|
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30010,South West,Male,25.0,70.0,163.0,175.0,135.0,240.0,220.0,I eat quality foods but don't measure the amou...,I have no athletic background besides CrossFit|,I began CrossFit with a coach (e.g. at an affi...,I typically rest fewer than 4 days per month|,1-2 years|
30011,Australia,Male,24.0,70.0,174.0,143.0,121.0,351.0,287.0,I eat 1-3 full cheat meals per week|,I played youth or high school level sports|I p...,I began CrossFit with a coach (e.g. at an affi...,I typically rest 4 or more days per month|,6-12 months|
30012,Latin America,Female,25.0,64.0,126.0,110.0,88.0,243.0,176.0,I eat quality foods but don't measure the amount|,I played youth or high school level sports|I p...,I began CrossFit by trying it alone (without a...,I usually only do 1 workout a day|I typically ...,Less than 6 months|
30013,North Central,Female,22.0,72.0,174.0,115.0,95.0,175.0,115.0,I eat quality foods but don't measure the amount|,I played youth or high school level sports|I p...,I began CrossFit with a coach (e.g. at an affi...,I usually only do 1 workout a day|I typically ...,Less than 6 months|


In [17]:
# Prepare for ML - SAME CODE as V1
X_train, X_test, y_train, y_test, df_full = prepare_for_ml(df)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Train: 24012, Test: 6003


In [37]:
# Train models - EXACTLY THE SAME CODE as V1
print("="*60)
print("DVC - TRAINING ON V2 (CLEANED DATA) - SAME CODE!")
print("="*60)

dvc_v2_results = {}

dvc_v2_results['rf'], _, _ = train_and_evaluate(
    X_train, X_test, y_train, y_test,
    RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE), "Random Forest"
)

DVC - TRAINING ON V2 (CLEANED DATA) - SAME CODE!

Random Forest:
  RMSE: 148.61
  MAE:  113.87
  R2:   0.7161


## DVC: Compare V1 vs V2

In [38]:
print("\n" + "="*70)
print("DVC - MODEL COMPARISON: V1 vs V2")
print("="*70)

for model_type in ['rf']:
    v1 = dvc_v1_results[model_type]
    v2 = dvc_v2_results[model_type]
    print(f"\n{v1['model']}:")
    print(f"  V1 RMSE: {v1['rmse']:.2f} -> V2 RMSE: {v2['rmse']:.2f} (Change: {v2['rmse']-v1['rmse']:+.2f})")
    print(f"  V1 R2:   {v1['r2']:.4f} -> V2 R2:   {v2['r2']:.4f} (Change: {v2['r2']-v1['r2']:+.4f})")


DVC - MODEL COMPARISON: V1 vs V2

Random Forest:
  V1 RMSE: 104087.33 -> V2 RMSE: 148.61 (Change: -103938.72)
  V1 R2:   -1.4852 -> V2 R2:   0.7161 (Change: +2.2013)


### Checkout back to v1

In [18]:
dvc_checkout("v1")

Switched to data version: v1


True

---
# ========================================================
# TOOL 2: Git-LFS
# ========================================================


## Git-LFS Setup

Install git-lfs
```bash
brew install git-lfs
```

Set it up in the repo
```bash
git lfs install
```

Track .csv files
```bash
git lfs track "*.csv"
```

Set up gitattributes and commit and push to remote
```bash
git add .gitattributes
git add *.csv
git commit -m "Track CSV files with Git LFS"
git push origin main
```

Verify tracking
```bash
git lfs ls-files
```

In [20]:
import subprocess
import os

def run_cmd(cmd):
    """Utility to run shell commands."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error: {result.stderr}")
    return result.stdout.strip()


def git_lfs_init():
    """Initialize Git LFS."""
    print(run_cmd("git lfs install"))


def git_lfs_track_csv():
    """Track CSV files with Git LFS."""
    print(run_cmd('git lfs track "*.csv"'))
    run_cmd("git add .gitattributes")
    print("Tracking .csv files with Git LFS")


def git_add_commit(message):
    """Add and commit changes."""
    run_cmd("git add .")
    print(run_cmd(f'git commit -m "{message}"'))


def git_push(branch="main"):
    """Push to remote."""
    print(run_cmd(f"git push origin {branch}"))


def git_lfs_version_data(file_path, version_tag):
    """
    Version a dataset (v1, v2, etc.)
    """
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return

    run_cmd(f"git add {file_path}")
    run_cmd(f'git commit -m "Add dataset version {version_tag}"')
    run_cmd(f"git tag {version_tag}")
    print(f"Versioned {file_path} as {version_tag}")


def git_checkout_version(version_tag):
    """Switch between dataset versions."""
    print(run_cmd(f"git checkout {version_tag}"))


def git_lfs_list_files():
    """List files tracked by Git LFS."""
    print(run_cmd("git lfs ls-files"))

## Git-LFS: Step 1 - Upload and Tag V1

Update and tag raw data with Git-LFS
```bash
git add "athletes.csv"
git commit -m "add V1 raw data"
git tag git-lfs_v1
git push origin main
```

In [21]:
import pandas as pd

print("Loading V1 from Git LFS...")

# Checkout the tagged version (v1)
git_checkout_version("git-lfs_v1")

df = pd.read_csv('athletes.csv')

print(f"Loaded data shape: {df.shape}")

Loading V1 from Git LFS...
M	Assignment_DataVersioning_DP.ipynb
Loaded data shape: (423006, 27)


In [22]:
# EDA V1
run_eda(df, "Git LFS v1")


EDA for Git LFS v1
Shape: (423006, 27)

Missing Values: 7643954

Numeric Statistics:
          athlete_id            age        height         weight  \
count  423003.000000  331110.000000  1.598690e+05  229890.000000   
mean   292748.166538      32.516750  1.206217e+02     170.896137   
std    184969.660327       7.730671  2.097995e+04      58.379799   
min        82.000000      13.000000  0.000000e+00       1.000000   
25%    135091.500000      27.000000  6.600000e+01     145.000000   
50%    275839.000000      31.000000  6.900000e+01     170.000000   
75%    473188.000000      37.000000  7.200000e+01     192.000000   
max    633083.000000     125.000000  8.388607e+06   20175.000000   

               fran         helen         grace      filthy50      fgonebad  \
count  5.542600e+04  3.027900e+04  4.074500e+04  1.935900e+04  2.973800e+04   
mean   9.886691e+02  1.207950e+03  5.766025e+02  2.127863e+03  1.472252e+03   
std    7.200430e+04  6.824091e+04  4.891145e+04  6.055021e+04  9

,athlete_id,name,region,team,affiliate,gender,age,height,weight,fran,...,snatch,deadlift,backsq,pullups,eat,train,background,experience,schedule,howlong
0,2554.0,Pj Ablang,South West,Double Edge,Double Edge CrossFit,Male,24.0,70.0,166.0,NaN,...,NaN,400.0,305.0,NaN,NaN,I workout mostly at a CrossFit Affiliate|I hav...,I played youth or high school level sports|I r...,I began CrossFit with a coach (e.g. at an affi...,I do multiple workouts in a day 2x a week|,4+ years|
1,3517.0,Derek Abdella,NaN,NaN,NaN,Male,42.0,70.0,190.0,NaN,...,NaN,NaN,NaN,NaN,NaN,I have a coach who determines my programming|I...,I played youth or high school level sports|,I began CrossFit with a coach (e.g. at an affi...,I do multiple workouts in a day 2x a week|,4+ years|
2,4691.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5164.0,Abo Brandon,Southern California,LAX CrossFit,LAX CrossFit,Male,40.0,67.0,NaN,211.0,...,200.0,375.0,325.0,25.0,I eat 1-3 full cheat meals per week|,I workout mostly at a CrossFit Affiliate|I hav...,I played youth or high school level sports|,I began CrossFit by trying it alone (without a...,I usually only do 1 workout a day|,4+ years|
4,5286.0,Bryce Abbey,NaN,NaN,NaN,Male,32.0,65.0,149.0,206.0,...,150.0,NaN,325.0,50.0,I eat quality foods but don't measure the amount|,I workout mostly at a CrossFit Affiliate|I inc...,I played college sports|,I began CrossFit by trying it alone (without a...,I usually only do 1 workout a day|I strictly s...,1-2 years|
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
423001,574489.0,Odo Renata,Latin America,Team Guarujá Inox,CrossFit Guaruja,Female,36.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
423002,585696.0,Lozzie Trevor,Australia,FBP CrossFit Games Team,FBP CrossFit,Female,27.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
423003,608828.0,Marisol Smith,North West,CrossFit Oak Harbor,CrossFit Oak Harbor,Female,44.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
423004,628881.0,Pedrini Morgane,Europe,NaN,CrossFit 67,Female,20.0,64.0,61.0,NaN,...,NaN,80.0,143.0,NaN,I eat quality foods but don't measure the amount|,I workout mostly at a CrossFit Affiliate|,NaN,I began CrossFit with a coach (e.g. at an affi...,I usually only do 1 workout a day|I strictly s...,6-12 months|


In [59]:
# Prepare and train on V1
df_v1_clean = df.dropna(subset=['deadlift', 'candj', 'snatch', 'backsq'])
df_v1_clean = df_v1_clean[(df_v1_clean['deadlift'] > 0) & 
                          (df_v1_clean['candj'] > 0) & 
                          (df_v1_clean['snatch'] > 0) & 
                          (df_v1_clean['backsq'] > 0)]

X_train, X_test, y_train, y_test, _ = prepare_for_ml(df_v1_clean)

print("="*60)
print("LAKEFS - TRAINING ON V1 (RAW DATA)")
print("="*60)

lakefs_v1_results = {}

lakefs_v1_results['rf'], _, _ = train_and_evaluate(
    X_train, X_test, y_train, y_test,
    RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE), "Random Forest"
)

LAKEFS - TRAINING ON V1 (RAW DATA)

Random Forest:
  RMSE: 104087.33
  MAE:  2013.66
  R2:   -1.4852


## Git LFS: Step 2 - Upload V2 (Cleaned)

In [23]:
# Clean the data and save as the SAME filename (important)
df_cleaned = clean_data(df)
df_cleaned.to_csv('athletes.csv', index=False)  # overwrite

print(f"Cleaned data shape: {df_cleaned.shape}")

Cleaned data shape: (30015, 14)


Update and tag clean data with Git-LFS
```bash
git add "athletes.csv"
git commit -m "add V2 clean data"
git tag git-lfs_v2
git push origin main
```

## Git-LFS: Step 3 - Switch to V2 and Train (SAME CODE)

In [26]:
print("Loading V2 from Git LFS...")

git_checkout_version("git-lfs_v2")  

df = pd.read_csv('athletes.csv')

print(f"Loaded data shape: {df.shape}")

Loading V2 from Git LFS...

Loaded data shape: (30015, 14)


In [27]:
# EDA V2 - SAME CODE
run_eda(df, "Git LFS V2 (Cleaned Data)")


EDA for Git LFS V2 (Cleaned Data)
Shape: (30015, 14)

Missing Values: 0

Numeric Statistics:
                age        height        weight         candj        snatch  \
count  30015.000000  30015.000000  30015.000000  30015.000000  30015.000000   
mean      32.125271     68.873397    177.146660    205.427520    156.178011   
std        7.436255      3.774809     32.397735     58.696774     48.841798   
min       18.000000     52.000000      5.000000      1.000000      1.000000   
25%       27.000000     66.000000    155.000000    160.000000    120.000000   
50%       31.000000     69.000000    178.000000    205.000000    155.000000   
75%       37.000000     72.000000    197.000000    245.000000    190.000000   
max       56.000000     83.000000    474.000000    390.000000    386.000000   

           deadlift        backsq  
count  30015.000000  30015.000000  
mean     362.102815    294.141596  
std       96.451618     85.090633  
min        1.000000      1.000000  
25%      287.0

,region,gender,age,height,weight,candj,snatch,deadlift,backsq,eat,background,experience,schedule,howlong
0,Southern California,Male,30.0,71.0,200.0,235.0,175.0,385.0,315.0,I eat whatever is convenient|,I played youth or high school level sports|I p...,I began CrossFit by trying it alone (without a...,I do multiple workouts in a day 1x a week|I ty...,1-2 years|
1,Africa,Male,28.0,70.0,176.0,187.0,134.0,335.0,254.0,I eat 1-3 full cheat meals per week|,I have no athletic background besides CrossFit|,I began CrossFit with a coach (e.g. at an affi...,I do multiple workouts in a day 1x a week|,2-4 years|
2,North East,Male,35.0,68.0,225.0,285.0,205.0,440.0,405.0,I eat quality foods but don't measure the amount|,I played youth or high school level sports|,I began CrossFit with a coach (e.g. at an affi...,I typically rest 4 or more days per month|,2-4 years|
3,North Central,Male,36.0,71.0,199.0,267.0,212.0,485.0,390.0,I eat quality foods but don't measure the amount|,I played youth or high school level sports|I p...,I began CrossFit with a coach (e.g. at an affi...,I do multiple workouts in a day 3+ times a wee...,1-2 years|
4,North East,Male,36.0,64.0,155.0,245.0,180.0,415.0,385.0,I eat strict Paleo|,I played youth or high school level sports|I p...,I began CrossFit by trying it alone (without a...,I do multiple workouts in a day 2x a week|I st...,4+ years|
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30010,South West,Male,25.0,70.0,163.0,175.0,135.0,240.0,220.0,I eat quality foods but don't measure the amou...,I have no athletic background besides CrossFit|,I began CrossFit with a coach (e.g. at an affi...,I typically rest fewer than 4 days per month|,1-2 years|
30011,Australia,Male,24.0,70.0,174.0,143.0,121.0,351.0,287.0,I eat 1-3 full cheat meals per week|,I played youth or high school level sports|I p...,I began CrossFit with a coach (e.g. at an affi...,I typically rest 4 or more days per month|,6-12 months|
30012,Latin America,Female,25.0,64.0,126.0,110.0,88.0,243.0,176.0,I eat quality foods but don't measure the amount|,I played youth or high school level sports|I p...,I began CrossFit by trying it alone (without a...,I usually only do 1 workout a day|I typically ...,Less than 6 months|
30013,North Central,Female,22.0,72.0,174.0,115.0,95.0,175.0,115.0,I eat quality foods but don't measure the amount|,I played youth or high school level sports|I p...,I began CrossFit with a coach (e.g. at an affi...,I usually only do 1 workout a day|I typically ...,Less than 6 months|


In [63]:
# Prepare and train - SAME CODE
X_train, X_test, y_train, y_test, _ = prepare_for_ml(df)

print("="*60)
print("Git-LFS - TRAINING ON V2 (CLEANED DATA)")
print("="*60)

lakefs_v2_results = {}

lakefs_v2_results['rf'], _, _ = train_and_evaluate(
    X_train, X_test, y_train, y_test,
    RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE), "Random Forest"
)

LAKEFS - TRAINING ON V2 (CLEANED DATA) - SAME CODE!

Random Forest:
  RMSE: 148.61
  MAE:  113.87
  R2:   0.7161


## Git-LFS: Compare V1 vs V2

In [65]:
print("\n" + "="*70)
print("Git-LFS - MODEL COMPARISON: V1 vs V2")
print("="*70)

for model_type in ['rf']:
    v1 = lakefs_v1_results[model_type]
    v2 = lakefs_v2_results[model_type]
    print(f"\n{v1['model']}:")
    print(f"  V1 RMSE: {v1['rmse']:.2f} -> V2 RMSE: {v2['rmse']:.2f} (Change: {v2['rmse']-v1['rmse']:+.2f})")
    print(f"  V1 R2:   {v1['r2']:.4f} -> V2 R2:   {v2['r2']:.4f} (Change: {v2['r2']-v1['r2']:+.4f})")


Git-LFS - MODEL COMPARISON: V1 vs V2

Random Forest:
  V1 RMSE: 104087.33 -> V2 RMSE: 148.61 (Change: -103938.72)
  V1 R2:   -1.4852 -> V2 R2:   0.7161 (Change: +2.2013)


---
# ========================================================
# Differential Privacy with Opacus (Using V2 Data)
# ========================================================

In [63]:
!{sys.executable} -m pip install opacus==1.4.0 
!{sys.executable} -m pip install "numpy<2"
!{sys.executable} -m pip install --upgrade torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu


In [64]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

try:
    from opacus import PrivacyEngine
    OPACUS_AVAILABLE = True
    print("Opacus available")
except ImportError:
    OPACUS_AVAILABLE = False
    print("Install Opacus: pip install opacus")

Opacus available


In [65]:
class RegressionNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.net(x)

In [66]:
# Prepare PyTorch data (using V2 - the currently loaded version)
def prepare_torch_data(X_train, X_test, y_train, y_test, batch_size=64):
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    train_ds = TensorDataset(
        torch.FloatTensor(X_train_s),
        torch.FloatTensor(y_train.values).unsqueeze(1)
    )
    test_ds = TensorDataset(
        torch.FloatTensor(X_test_s),
        torch.FloatTensor(y_test.values).unsqueeze(1)
    )
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)
    
    return train_loader, test_loader, X_train_s.shape[1]

train_loader, test_loader, input_dim = prepare_torch_data(X_train, X_test, y_train, y_test)
print(f"Input dim: {input_dim}")

Input dim: 10


In [69]:
def train_pytorch(model, train_loader, epochs=50):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

def evaluate_pytorch(model, test_loader):
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

    model.eval()
    preds, actuals = [], []

    with torch.no_grad():
        for X, y in test_loader:
            outputs = model(X).squeeze()
            preds.extend(outputs.detach().cpu().tolist())  # <- use .tolist()
            actuals.extend(y.detach().cpu().tolist())

    return {
        'rmse': np.sqrt(mean_squared_error(actuals, preds)),
        'mae': mean_absolute_error(actuals, preds),
        'r2': r2_score(actuals, preds)
    }

In [70]:
# Train Non-DP Model
print("="*60)
print("Training NON-DP Neural Network on V2")
print("="*60)

model_non_dp = RegressionNet(input_dim)
train_pytorch(model_non_dp, train_loader, epochs=50)
metrics_non_dp = evaluate_pytorch(model_non_dp, test_loader)

print(f"\nNon-DP Results: RMSE={metrics_non_dp['rmse']:.2f}, R2={metrics_non_dp['r2']:.4f}")

Training NON-DP Neural Network on V2
Epoch 10/50, Loss: 21876.8438
Epoch 20/50, Loss: 20058.8301
Epoch 30/50, Loss: 23889.2656
Epoch 40/50, Loss: 22223.6484
Epoch 50/50, Loss: 9946.5283

Non-DP Results: RMSE=163.11, R2=0.6580


In [71]:
# Train DP Model
if OPACUS_AVAILABLE:
    print("\n" + "="*60)
    print("Training DP Neural Network on V2 (Opacus)")
    print("="*60)
    
    EPOCHS = 50
    EPSILON = 10.0
    DELTA = 1e-5
    MAX_GRAD_NORM = 1.0
    
    model_dp = RegressionNet(input_dim)
    optimizer_dp = optim.Adam(model_dp.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    
    privacy_engine = PrivacyEngine()
    model_dp, optimizer_dp, train_loader_dp = privacy_engine.make_private_with_epsilon(
        module=model_dp,
        optimizer=optimizer_dp,
        data_loader=train_loader,
        epochs=EPOCHS,
        target_epsilon=EPSILON,
        target_delta=DELTA,
        max_grad_norm=MAX_GRAD_NORM,
    )
    
    print(f"Noise multiplier: {optimizer_dp.noise_multiplier:.4f}")
    
    for epoch in range(EPOCHS):
        model_dp.train()
        for X_batch, y_batch in train_loader_dp:
            optimizer_dp.zero_grad()
            loss = criterion(model_dp(X_batch), y_batch)
            loss.backward()
            optimizer_dp.step()
        
        if (epoch + 1) % 10 == 0:
            eps = privacy_engine.get_epsilon(delta=DELTA)
            print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {loss.item():.4f}, ε: {eps:.2f}")
    
    final_epsilon = privacy_engine.get_epsilon(delta=DELTA)
    print(f"\nFinal Privacy Budget: ε={final_epsilon:.2f}, δ={DELTA}")
    
    metrics_dp = evaluate_pytorch(model_dp, test_loader)
    print(f"DP Results: RMSE={metrics_dp['rmse']:.2f}, R2={metrics_dp['r2']:.4f}")
else:
    metrics_dp = None
    final_epsilon = None


Training DP Neural Network on V2 (Opacus)
Noise multiplier: 0.5566
Epoch 10/50, Loss: 42185.3594, ε: 5.22
Epoch 20/50, Loss: 31003.1738, ε: 6.69
Epoch 30/50, Loss: 37508.2031, ε: 7.91
Epoch 40/50, Loss: 27214.8711, ε: 9.00
Epoch 50/50, Loss: 24013.5488, ε: 10.00

Final Privacy Budget: ε=10.00, δ=1e-05
DP Results: RMSE=169.40, R2=0.6311


---
# Comparison: Non-DP vs DP

In [72]:
print("\n" + "="*70)
print("NON-DP vs DP MODEL COMPARISON (V2 Data)")
print("="*70)

if metrics_dp:
    comparison = pd.DataFrame({
        'Metric': ['RMSE', 'MAE', 'R2', 'Privacy (ε)'],
        'Non-DP': [f"{metrics_non_dp['rmse']:.2f}", f"{metrics_non_dp['mae']:.2f}", 
                   f"{metrics_non_dp['r2']:.4f}", 'None'],
        'DP': [f"{metrics_dp['rmse']:.2f}", f"{metrics_dp['mae']:.2f}", 
               f"{metrics_dp['r2']:.4f}", f"{final_epsilon:.2f}"]
    })
    print(comparison.to_string(index=False))
    
    print(f"\nAccuracy trade-off for privacy:")
    print(f"  RMSE increase: {((metrics_dp['rmse']-metrics_non_dp['rmse'])/metrics_non_dp['rmse']*100):.1f}%")
    print(f"  R2 decrease: {((metrics_non_dp['r2']-metrics_dp['r2'])/metrics_non_dp['r2']*100):.1f}%")


NON-DP vs DP MODEL COMPARISON (V2 Data)
     Metric Non-DP     DP
       RMSE 163.11 169.40
        MAE 126.30 131.18
         R2 0.6580 0.6311
Privacy (ε)   None  10.00

Accuracy trade-off for privacy:
  RMSE increase: 3.9%
  R2 decrease: 4.1%


---
# Part B: Data Lineage

Use Marquez or dbt-core to create data lineage diagram.

In [ ]:
print("""
DATA LINEAGE DIAGRAM:

┌─────────────────┐
│   athletes.csv  │  <- Raw Source
│   (V1 - Raw)    │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│ TRANSFORMATIONS │
│ - Drop NaN      │
│ - Remove cols   │
│ - Filter rows   │
│ - Clean surveys │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  athletes.csv   │  <- Same file, new version
│  (V2 - Cleaned) │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│ FEATURE ENG     │
│ - total_lift    │
│ - encode cats   │
└────────┬────────┘
         │
    ┌────┴────┐
    ▼         ▼
┌───────┐ ┌───────┐
│ TRAIN │ │ TEST  │
│  80%  │ │  20%  │
└───────┘ └───────┘
""")

---
# Summary for Slides

In [ ]:
print("\n" + "="*80)
print("SLIDE 1: Non-DP vs DP Comparison")
print("="*80)
if metrics_dp:
    print(f"""
| Model  | RMSE  | R2     | Privacy |
|--------|-------|--------|----------|
| Non-DP | {metrics_non_dp['rmse']:.2f} | {metrics_non_dp['r2']:.4f} | None     |
| DP     | {metrics_dp['rmse']:.2f} | {metrics_dp['r2']:.4f} | ε={final_epsilon:.2f}   |

Key Insight: DP provides privacy guarantee at cost of ~{((metrics_dp['rmse']-metrics_non_dp['rmse'])/metrics_non_dp['rmse']*100):.0f}% accuracy
""")

print("\n" + "="*80)
print("SLIDE 2: Tool Comparison")
print("="*80)
print("""
| Criteria          | DVC           | LakeFS        |
|-------------------|---------------|---------------|
| Installation      | Easy (pip)    | Medium (server)|
| Version Switch    | CLI commands  | API parameter |
| Best For          | ML experiments| Production    |

Recommendation: DVC for this assignment, LakeFS for enterprise.
""")